# Steam GNN Recommendation System

End-to-end link prediction on Steam purchase data. Compares classical graph heuristics against a heterogeneous GraphSAGE model.

In [ ]:
import torch
print("PyTorch has version {}".format(torch.__version__))

torch_version = str(torch.__version__)
pyg_whl = f"https://data.pyg.org/whl/torch-{torch_version}.html"

!pip uninstall -y torch-geometric torch-sparse torch-scatter torch-cluster torch-spline-conv pyg-lib
!pip install torch-scatter -f $pyg_whl -q
!pip install torch-sparse  -f $pyg_whl -q
!pip install pyg-lib       -f $pyg_whl -q
!pip install torch-geometric -q


## 1. Setup

Install PyG and import libraries.

In [ ]:
import ast
import json
import numpy as np
import scipy.sparse as sp
from collections import defaultdict
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import normalize
import kagglehub

import torch
import torch.nn.functional as F
from torch import Tensor
import torch_geometric.transforms as T
from torch_geometric.data import HeteroData
from torch_geometric.loader import LinkNeighborLoader
from torch_geometric.metrics import LinkPredRecall, LinkPredNDCG, LinkPredMRR
from torch_geometric.nn import SAGEConv, to_hetero
from torch_geometric.utils import coalesce

import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")


In [ ]:
DATA_PATH = kagglehub.dataset_download(
    "pypiahmad/steam-video-game-and-bundle-data",
    path="australian_users_items.json"
)


## 2. Data Loading

Download the Steam dataset and build the bipartite user–item graph.

In [ ]:
K_VAL = 20
RANDOM_SEED = 42

def create_pyg_graph(file_path):
    data = HeteroData()

    user_map = {}
    item_map = {}

    item_names = []

    sources = []
    targets = []

    with open(file_path, encoding='utf-8') as f:
      for line in f:
          user = ast.literal_eval(line)
          u_id = str(user['user_id'])

          if u_id not in user_map:
              user_map[u_id] = len(user_map)
          u_idx = user_map[u_id]

          for item in user['items']:
              i_id = str(item.get('item_id'))

              if i_id not in item_map:
                  item_map[i_id] = len(item_map)
                  item_names.append(item.get('item_name', 'Unknown'))

              i_idx = item_map[i_id]

              sources.append(u_idx)
              targets.append(i_idx)

    num_users = len(user_map)
    num_items = len(item_map)

    data['user'].num_nodes = num_users
    data['item'].num_nodes = num_items

    data['user'].node_id = torch.arange(num_users)
    data['item'].node_id = torch.arange(num_items)

    edge_index = torch.tensor([sources, targets], dtype=torch.long)
    edge_index = coalesce(edge_index)
    data['user', 'purchased', 'item'].edge_index = edge_index

    data = T.ToUndirected()(data)

    print(f" users: {num_users}")
    print(f" items: {num_items}")
    return data, item_names, user_map, item_map, num_users, num_items


data, item_names, user_map, item_map, num_users, num_items = create_pyg_graph(DATA_PATH)


## 3. Data Splits

Split edges into train / val / test. Within training: 70% are message-passing edges, 30% are supervision edges (disjoint to avoid data leakage).

In [ ]:

transform = T.RandomLinkSplit(
    num_val=0.1,
    num_test=0.1,
    disjoint_train_ratio=0.3,
    neg_sampling_ratio=2.0,
    add_negative_train_samples=False,
    edge_types=("user", "purchased", "item"),
    rev_edge_types=("item", "rev_purchased", "user"),
)
train_data, val_data, test_data = transform(data)

train_sup_index  = train_data["user", "purchased", "item"].edge_label_index
train_sup_labels = train_data["user", "purchased", "item"].edge_label
train_pos_edges  = train_sup_index[:, train_sup_labels == 1]

# All training edges (MP + supervision) — gives baselines the same 80% the GNN trains on
train_full_edges = torch.cat([
    train_data["user", "purchased", "item"].edge_index,  # 56% MP edges
    train_pos_edges                                       # 24% supervision edges
], dim=1)


## 4. Evaluation Helpers

In [ ]:
def run_evaluation(val_split, train_sparse_matrix, score_matrix, k=20):
    edge_index = val_split["user", "purchased", "item"].edge_label_index
    edge_label = val_split["user", "purchased", "item"].edge_label
    pos_mask = (edge_label == 1)

    # Use bool mask to avoid densifying a float64 sparse matrix (~8x memory overhead)
    train_mask = torch.from_numpy(train_sparse_matrix.toarray().astype(np.bool_))
    scores = torch.from_numpy(score_matrix)
    if scores.dtype != torch.float32:
        scores = scores.float()
    scores[train_mask] = -float('inf')
    del train_mask
    _, topk_indices = torch.topk(scores, k=k, dim=1)
    del scores

    recall_metric = LinkPredRecall(k=k)
    ndcg_metric   = LinkPredNDCG(k=k)
    mrr_metric    = LinkPredMRR(k=k)

    pos_edge_index = edge_index[:, pos_mask]
    recall_metric.update(pred_index_mat=topk_indices, edge_label_index=pos_edge_index)
    ndcg_metric.update(pred_index_mat=topk_indices,   edge_label_index=pos_edge_index)
    mrr_metric.update(pred_index_mat=topk_indices,    edge_label_index=pos_edge_index)

    val_users = edge_index[0].numpy()
    val_items = edge_index[1].numpy()
    raw = torch.from_numpy(score_matrix)
    if raw.dtype != torch.float32:
        raw = raw.float()
    val_predictions = raw[val_users, val_items].numpy()
    val_auc = roc_auc_score(edge_label.numpy(), val_predictions)

    return (
        recall_metric.compute().item(),
        ndcg_metric.compute().item(),
        mrr_metric.compute().item(),
        val_auc
    )


## 5. Baseline Methods

Classical graph link-prediction heuristics: Preferential Attachment, Jaccard, Resource Allocation, Heat Conduction.

In [ ]:
class ClassicalGraphPredictor:

    def __init__(self, num_users, num_items, method='resource_allocation'):
        self.num_users = num_users
        self.num_items = num_items
        self.method = method
        self.R_train = None
        self.user_degrees = None
        self.item_degrees = None
        self.projection_matrix = None

    def fit(self, edge_index):
        self.R_train = sp.csr_matrix(
            (np.ones(edge_index.shape[1]),
             (edge_index[0].numpy(), edge_index[1].numpy())),
            shape=(self.num_users, self.num_items)
        )
        self.user_degrees = np.array(self.R_train.sum(axis=1)).flatten()
        self.item_degrees = np.array(self.R_train.sum(axis=0)).flatten()

        if self.method == 'preferential_attachment':
            return
        elif self.method == 'jaccard':
            co = (self.R_train.T @ self.R_train).tocsr()
            rows, cols = co.nonzero()
            co_vals = np.asarray(co[rows, cols]).flatten()
            union = self.item_degrees[rows] + self.item_degrees[cols] - co_vals
            jaccard_vals = np.divide(
                co_vals, union,
                out=np.zeros_like(co_vals, dtype=float),
                where=union > 0
            )
            self.projection_matrix = sp.csr_matrix(
                (jaccard_vals, (rows, cols)), shape=co.shape
            )
        elif self.method == 'resource_allocation':
            W_user = normalize(self.R_train, norm='l1', axis=1)
            self.projection_matrix = (self.R_train.T @ W_user).tocsr()
        elif self.method == 'heat_conduction':
            W_user = normalize(self.R_train, norm='l1', axis=1)
            W_item = normalize(self.R_train, norm='l1', axis=0)
            self.projection_matrix = (W_item.T @ W_user).tocsr()

    def predict_chunk(self, user_indices):
        """Score all items for a subset of users. Returns float32 (n_users, n_items)."""
        if self.method == 'preferential_attachment':
            return (self.user_degrees[user_indices, None] * self.item_degrees[None, :]).astype(np.float32)
        return (self.R_train[user_indices] @ self.projection_matrix).toarray().astype(np.float32)


## 6. GNN Architecture

Heterogeneous GraphSAGE with separate learnable embeddings for users and items.

In [ ]:
class GNN(torch.nn.Module):
    def __init__(self, hidden_channels):
        super().__init__()
        self.conv1 = SAGEConv(hidden_channels, hidden_channels)
        self.conv2 = SAGEConv(hidden_channels, hidden_channels)

    def forward(self, x: Tensor, edge_index: Tensor) -> Tensor:
        x = F.relu(self.conv1(x, edge_index))
        x = self.conv2(x, edge_index)
        return x


class Classifier(torch.nn.Module):
    def forward(self, x_user: Tensor, x_item: Tensor, edge_label_index: Tensor) -> Tensor:
        edge_feat_user = x_user[edge_label_index[0]]
        edge_feat_item = x_item[edge_label_index[1]]
        return (edge_feat_user * edge_feat_item).sum(dim=-1)


class Model(torch.nn.Module):
    def __init__(self, hidden_channels):
        super().__init__()
        self.item_lin = torch.nn.Linear(20, hidden_channels)
        self.user_emb = torch.nn.Embedding(data["user"].num_nodes, hidden_channels)
        self.item_emb = torch.nn.Embedding(data["item"].num_nodes, hidden_channels)
        self.gnn = to_hetero(GNN(hidden_channels), metadata=data.metadata())
        self.classifier = Classifier()

    def forward(self, data: HeteroData) -> Tensor:
        x_dict = {
            "user": self.user_emb(data["user"].node_id),
            "item": self.item_emb(data["item"].node_id)
        }
        x_dict = self.gnn(x_dict, data.edge_index_dict)
        return self.classifier(
            x_dict["user"],
            x_dict["item"],
            data["user", "purchased", "item"].edge_label_index
        )

    @torch.no_grad()
    def get_user_item_embeddings(self, train_data):
        """Returns (user_emb, item_emb) as float32 numpy arrays. Small (~25 MB total)."""
        self.eval()
        edge_index_dict = {k: v.to(device) for k, v in train_data.edge_index_dict.items()}
        x_dict = {
            "user": self.user_emb(data["user"].node_id.to(device)),
            "item": self.item_emb(data["item"].node_id.to(device))
        }
        x_dict = self.gnn(x_dict, edge_index_dict)
        return (
            x_dict["user"].cpu().numpy().astype(np.float32),
            x_dict["item"].cpu().numpy().astype(np.float32)
        )

    @torch.no_grad()
    def get_score_matrix(self, train_data):
        user_emb, item_emb = self.get_user_item_embeddings(train_data)
        return (user_emb @ item_emb.T).astype(np.float32)


## 7. Training

### 7a. BCE Loss (5 Epochs)

Train with Binary Cross-Entropy. Loads from checkpoint if already saved.

In [ ]:
import os
import matplotlib.pyplot as plt

HIDDEN_CHANNELS = 64
EPOCHS = 5
LR = 0.001
BATCH_SIZE = 128
MODEL_PATH = '/content/gnn_model.pth'


@torch.no_grad()
def estimate_val_loss(model, n_samples=8000):
    """BCE loss on a random sample of validation edges — fast overfitting proxy."""
    model.eval()
    ei = val_data["user", "purchased", "item"].edge_label_index
    el = val_data["user", "purchased", "item"].edge_label.float()
    idx = torch.randperm(ei.shape[1])[:n_samples]
    ei_s, el_s = ei[:, idx].to(device), el[idx].to(device)
    edge_index_dict = {k: v.to(device) for k, v in train_data.edge_index_dict.items()}
    x_dict = {
        "user": model.user_emb(data["user"].node_id.to(device)),
        "item": model.item_emb(data["item"].node_id.to(device))
    }
    x_dict = model.gnn(x_dict, edge_index_dict)
    scores = model.classifier(x_dict["user"], x_dict["item"], ei_s)
    return F.binary_cross_entropy_with_logits(scores, el_s).item()


train_losses, val_losses = [], []

if os.path.exists(MODEL_PATH):
    print(f"Found {MODEL_PATH} — loading, skipping training.")
    gnn_model = Model(hidden_channels=HIDDEN_CHANNELS).to(device)
    gnn_model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
    gnn_model.eval()

else:
    print("No saved model found — training from scratch...")
    gnn_model = Model(hidden_channels=HIDDEN_CHANNELS).to(device)
    optimizer = torch.optim.Adam(gnn_model.parameters(), lr=LR)

    train_loader = LinkNeighborLoader(
        data=train_data,
        num_neighbors=[20, 10],
        neg_sampling_ratio=2.0,
        edge_label_index=(("user", "purchased", "item"), train_data["user", "purchased", "item"].edge_label_index),
        edge_label=train_data["user", "purchased", "item"].edge_label,
        batch_size=BATCH_SIZE,
        shuffle=True,
    )

    for epoch in range(1, EPOCHS + 1):
        total_loss = total_examples = 0
        gnn_model.train()
        for sampled_data in tqdm.tqdm(train_loader, desc=f"Epoch {epoch:03d}"):
            optimizer.zero_grad()
            sampled_data.to(device)
            pred = gnn_model(sampled_data)
            ground_truth = sampled_data["user", "purchased", "item"].edge_label
            loss = F.binary_cross_entropy_with_logits(pred, ground_truth)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * pred.numel()
            total_examples += pred.numel()

        avg_train = total_loss / total_examples
        avg_val = estimate_val_loss(gnn_model)
        train_losses.append(avg_train)
        val_losses.append(avg_val)
        print(f"Epoch {epoch:03d}: Train={avg_train:.4f}  Val={avg_val:.4f}")

    torch.save(gnn_model.state_dict(), MODEL_PATH)
    print(f"Saved to {MODEL_PATH}")

    plt.figure(figsize=(7, 4))
    plt.plot(range(1, EPOCHS + 1), train_losses, marker='o', label='Train')
    plt.plot(range(1, EPOCHS + 1), val_losses,   marker='o', label='Val')
    plt.xlabel('Epoch'); plt.ylabel('BCE Loss')
    plt.title('BCE: Train vs Val Loss')
    plt.legend(); plt.grid(True); plt.tight_layout()
    plt.savefig('bce_loss_curve.png', dpi=150)
    plt.show()


### 7b. BPR Loss Variant (5 Epochs)

Bayesian Personalized Ranking: optimises ranking order directly. Compare train/val loss curves against BCE above.

In [ ]:
# ── BPR Loss experiment — 5 epochs, fresh model ──────────────────────────────
# BPR loss: -mean( log σ(score_pos - score_neg) )
# Directly optimizes ranking order rather than binary classification.
# We run 5 epochs to compare convergence and overfitting vs BCE above.

BPR_MODEL_PATH = '/content/gnn_model_bpr.pth'


def bpr_loss(pred, labels):
    pos_scores = pred[labels == 1]
    neg_scores = pred[labels == 0]
    n_pos = pos_scores.shape[0]
    if n_pos == 0:
        return pred.sum() * 0
    neg_per_pos = neg_scores.shape[0] // n_pos
    # Pair each positive with its neg_per_pos negatives
    pos_exp = pos_scores.repeat_interleave(neg_per_pos)
    return -F.logsigmoid(pos_exp - neg_scores[:n_pos * neg_per_pos]).mean()


@torch.no_grad()
def estimate_val_loss_bpr(model, n_samples=8000):
    model.eval()
    ei = val_data["user", "purchased", "item"].edge_label_index
    el = val_data["user", "purchased", "item"].edge_label.float()
    idx = torch.randperm(ei.shape[1])[:n_samples]
    ei_s, el_s = ei[:, idx].to(device), el[idx].to(device)
    edge_index_dict = {k: v.to(device) for k, v in train_data.edge_index_dict.items()}
    x_dict = {
        "user": model.user_emb(data["user"].node_id.to(device)),
        "item": model.item_emb(data["item"].node_id.to(device))
    }
    x_dict = model.gnn(x_dict, edge_index_dict)
    scores = model.classifier(x_dict["user"], x_dict["item"], ei_s)
    return bpr_loss(scores, el_s).item()


bpr_train_losses, bpr_val_losses = [], []

if os.path.exists(BPR_MODEL_PATH):
    print(f"Found {BPR_MODEL_PATH} — loading, skipping BPR training.")
    gnn_bpr = Model(hidden_channels=HIDDEN_CHANNELS).to(device)
    gnn_bpr.load_state_dict(torch.load(BPR_MODEL_PATH, map_location=device))
    gnn_bpr.eval()

else:
    print("Training BPR model (5 epochs)...")
    gnn_bpr = Model(hidden_channels=HIDDEN_CHANNELS).to(device)
    optimizer_bpr = torch.optim.Adam(gnn_bpr.parameters(), lr=LR)

    bpr_loader = LinkNeighborLoader(
        data=train_data,
        num_neighbors=[20, 10],
        neg_sampling_ratio=2.0,
        edge_label_index=(("user", "purchased", "item"), train_data["user", "purchased", "item"].edge_label_index),
        edge_label=train_data["user", "purchased", "item"].edge_label,
        batch_size=BATCH_SIZE,
        shuffle=True,
    )

    for epoch in range(1, 6):
        total_loss = total_examples = 0
        gnn_bpr.train()
        for sampled_data in tqdm.tqdm(bpr_loader, desc=f"BPR Epoch {epoch:03d}"):
            optimizer_bpr.zero_grad()
            sampled_data.to(device)
            pred = gnn_bpr(sampled_data)
            labels = sampled_data["user", "purchased", "item"].edge_label
            loss = bpr_loss(pred, labels)
            loss.backward()
            optimizer_bpr.step()
            total_loss += loss.item() * pred.numel()
            total_examples += pred.numel()

        avg_train = total_loss / total_examples
        avg_val = estimate_val_loss_bpr(gnn_bpr)
        bpr_train_losses.append(avg_train)
        bpr_val_losses.append(avg_val)
        print(f"BPR Epoch {epoch:03d}: Train={avg_train:.4f}  Val={avg_val:.4f}")

    torch.save(gnn_bpr.state_dict(), BPR_MODEL_PATH)
    print(f"Saved to {BPR_MODEL_PATH}")

# ── Compare BCE vs BPR loss curves ───────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, t_l, v_l, title in [
    (axes[0], train_losses,     val_losses,     'BCE Loss'),
    (axes[1], bpr_train_losses, bpr_val_losses, 'BPR Loss'),
]:
    if t_l:
        ep = range(1, len(t_l) + 1)
        ax.plot(ep, t_l, marker='o', label='Train')
        ax.plot(ep, v_l, marker='o', label='Val')
    else:
        ax.text(0.5, 0.5, 'loaded from file\n(no curve data)',
                ha='center', va='center', transform=ax.transAxes)
    ax.set_title(title); ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
    ax.legend(); ax.grid(True)

plt.tight_layout()
plt.savefig('loss_comparison.png', dpi=150)
plt.show()
print("\nTo evaluate BPR on recommendation metrics, run cell-10 after setting:")
print("  gnn_model = gnn_bpr")


### 7c. BCE + Playtime Features (5 Epochs)

Augments the base model with per-user and per-item playtime statistics (mean log-playtime and log-degree) projected into the embedding space and added to the learnable embeddings.

In [ ]:
PLAYTIME_MODEL_PATH = '/content/gnn_model_playtime.pth'

# ── Compute per-node playtime statistics from raw data ─────────────────────
print("Computing playtime features...")
user_pt_sum   = np.zeros(num_users, dtype=np.float32)
user_pt_count = np.zeros(num_users, dtype=np.float32)
item_pt_sum   = np.zeros(num_items, dtype=np.float32)
item_pt_count = np.zeros(num_items, dtype=np.float32)

with open(DATA_PATH, encoding='utf-8') as f:
    for line in f:
        user = ast.literal_eval(line)
        u_id = str(user['user_id'])
        if u_id not in user_map:
            continue
        u_idx = user_map[u_id]
        for item in user['items']:
            i_id = str(item.get('item_id'))
            if i_id not in item_map:
                continue
            i_idx = item_map[i_id]
            lpt = np.log1p(item.get('playtime_forever', 0))
            user_pt_sum[u_idx]   += lpt;  user_pt_count[u_idx] += 1
            item_pt_sum[i_idx]   += lpt;  item_pt_count[i_idx] += 1

user_mean_pt = np.divide(user_pt_sum, user_pt_count, out=np.zeros_like(user_pt_sum), where=user_pt_count > 0)
item_mean_pt = np.divide(item_pt_sum, item_pt_count, out=np.zeros_like(item_pt_sum), where=item_pt_count > 0)

# Feature vectors: [mean_log_playtime, log_num_interactions]
user_feats_pt = torch.tensor(np.stack([user_mean_pt, np.log1p(user_pt_count)], axis=1), dtype=torch.float32)
item_feats_pt = torch.tensor(np.stack([item_mean_pt, np.log1p(item_pt_count)], axis=1), dtype=torch.float32)
print(f"User features: {user_feats_pt.shape},  Item features: {item_feats_pt.shape}")


# ── Model: learnable embedding + projected playtime features ───────────────
class ModelWithPlaytime(torch.nn.Module):
    def __init__(self, hidden_channels, user_feats, item_feats):
        super().__init__()
        self.user_emb      = torch.nn.Embedding(data["user"].num_nodes, hidden_channels)
        self.item_emb      = torch.nn.Embedding(data["item"].num_nodes, hidden_channels)
        self.user_feat_lin = torch.nn.Linear(user_feats.shape[1], hidden_channels)
        self.item_feat_lin = torch.nn.Linear(item_feats.shape[1], hidden_channels)
        self.register_buffer('user_feats', user_feats)
        self.register_buffer('item_feats', item_feats)
        self.gnn        = to_hetero(GNN(hidden_channels), metadata=data.metadata())
        self.classifier = Classifier()

    def forward(self, data: HeteroData) -> Tensor:
        uid, iid = data["user"].node_id, data["item"].node_id
        x_dict = {
            "user": self.user_emb(uid) + self.user_feat_lin(self.user_feats[uid]),
            "item": self.item_emb(iid) + self.item_feat_lin(self.item_feats[iid]),
        }
        x_dict = self.gnn(x_dict, data.edge_index_dict)
        return self.classifier(x_dict["user"], x_dict["item"],
                               data["user", "purchased", "item"].edge_label_index)

    @torch.no_grad()
    def get_user_item_embeddings(self, train_data):
        self.eval()
        edge_index_dict = {k: v.to(device) for k, v in train_data.edge_index_dict.items()}
        x_dict = {
            "user": self.user_emb(data["user"].node_id.to(device)) + self.user_feat_lin(self.user_feats),
            "item": self.item_emb(data["item"].node_id.to(device)) + self.item_feat_lin(self.item_feats),
        }
        x_dict = self.gnn(x_dict, edge_index_dict)
        return (x_dict["user"].cpu().numpy().astype(np.float32),
                x_dict["item"].cpu().numpy().astype(np.float32))


@torch.no_grad()
def estimate_val_loss_pt(model, n_samples=8000):
    """Val loss via full embedding pass — correct for any model with get_user_item_embeddings."""
    model.eval()
    ei = val_data["user", "purchased", "item"].edge_label_index
    el = val_data["user", "purchased", "item"].edge_label.float()
    idx = torch.randperm(ei.shape[1])[:n_samples]
    ei_s, el_s = ei[:, idx], el[idx].to(device)
    u_emb, i_emb = model.get_user_item_embeddings(train_data)
    u_t = torch.from_numpy(u_emb).to(device)
    i_t = torch.from_numpy(i_emb).to(device)
    scores = (u_t[ei_s[0]] * i_t[ei_s[1]]).sum(dim=-1)
    return F.binary_cross_entropy_with_logits(scores, el_s).item()


# ── Train ──────────────────────────────────────────────────────────────────
pt_train_losses, pt_val_losses = [], []

if os.path.exists(PLAYTIME_MODEL_PATH):
    print(f"Found {PLAYTIME_MODEL_PATH} — loading, skipping training.")
    gnn_playtime = ModelWithPlaytime(HIDDEN_CHANNELS, user_feats_pt, item_feats_pt).to(device)
    gnn_playtime.load_state_dict(torch.load(PLAYTIME_MODEL_PATH, map_location=device))
    gnn_playtime.eval()

else:
    print("Training playtime-augmented model (5 epochs)...")
    gnn_playtime = ModelWithPlaytime(HIDDEN_CHANNELS, user_feats_pt, item_feats_pt).to(device)
    optimizer_pt = torch.optim.Adam(gnn_playtime.parameters(), lr=LR)

    pt_loader = LinkNeighborLoader(
        data=train_data,
        num_neighbors=[20, 10],
        neg_sampling_ratio=2.0,
        edge_label_index=(("user", "purchased", "item"), train_data["user", "purchased", "item"].edge_label_index),
        edge_label=train_data["user", "purchased", "item"].edge_label,
        batch_size=BATCH_SIZE,
        shuffle=True,
    )

    for epoch in range(1, EPOCHS + 1):
        total_loss = total_examples = 0
        gnn_playtime.train()
        for sampled_data in tqdm.tqdm(pt_loader, desc=f"Playtime Epoch {epoch:03d}"):
            optimizer_pt.zero_grad()
            sampled_data.to(device)
            pred = gnn_playtime(sampled_data)
            ground_truth = sampled_data["user", "purchased", "item"].edge_label
            loss = F.binary_cross_entropy_with_logits(pred, ground_truth)
            loss.backward()
            optimizer_pt.step()
            total_loss += loss.item() * pred.numel()
            total_examples += pred.numel()

        avg_train = total_loss / total_examples
        avg_val   = estimate_val_loss_pt(gnn_playtime)
        pt_train_losses.append(avg_train)
        pt_val_losses.append(avg_val)
        print(f"Epoch {epoch:03d}: Train={avg_train:.4f}  Val={avg_val:.4f}")

    torch.save(gnn_playtime.state_dict(), PLAYTIME_MODEL_PATH)
    print(f"Saved to {PLAYTIME_MODEL_PATH}")

    plt.figure(figsize=(7, 4))
    plt.plot(range(1, EPOCHS + 1), pt_train_losses, marker='o', label='Train')
    plt.plot(range(1, EPOCHS + 1), pt_val_losses,   marker='o', label='Val')
    plt.xlabel('Epoch'); plt.ylabel('BCE Loss')
    plt.title('Playtime Features: Train vs Val Loss')
    plt.legend(); plt.grid(True); plt.tight_layout()
    plt.savefig('playtime_loss_curve.png', dpi=150)
    plt.show()


## 8. Evaluation

Chunked evaluation (500 users at a time) to stay within RAM limits. Reports Recall@20, NDCG@20, MRR@20, AUC for all methods.

In [ ]:
import gc

CHUNK_SIZE = 500  # users per chunk to save on RAM when running in google colab

def chunked_evaluate(val_data, score_fn, mask_sparse, user_degrees, k=K_VAL):
    """Evaluate one recommender method chunk-by-chunk to stay within RAM limits."""
    edge_index = val_data["user", "purchased", "item"].edge_label_index
    edge_label = val_data["user", "purchased", "item"].edge_label
    pos_ei = edge_index[:, edge_label == 1]
    ei0 = edge_index[0].numpy()
    pos_ei0 = pos_ei[0].numpy()
    el_np = edge_label.numpy()
    ei1_np = edge_index[1].numpy()

    all_users = torch.unique(pos_ei[0]).numpy()

    g_recall = LinkPredRecall(k=k)
    g_ndcg   = LinkPredNDCG(k=k)
    g_mrr    = LinkPredMRR(k=k)
    auc_preds, auc_labels = [], []

    buckets = [(-1, 0), (0, 1), (1, 5), (5, 10), (10, 50), (50, float('inf'))]
    b_recall = [LinkPredRecall(k=k) for _ in buckets]
    b_ndcg   = [LinkPredNDCG(k=k) for _ in buckets]
    b_mrr    = [LinkPredMRR(k=k) for _ in buckets]
    b_counts = [0] * len(buckets)

    for start in tqdm.tqdm(range(0, len(all_users), CHUNK_SIZE), desc="chunks"):
        chunk = all_users[start:start + CHUNK_SIZE]
        g2l = {int(u): i for i, u in enumerate(chunk)}

        scores_np = score_fn(chunk)  # float32, shape (chunk, num_items)

        # Collect raw scores for AUC before masking
        in_chunk = np.isin(ei0, chunk)
        if np.any(in_chunk):
            vu = ei0[in_chunk]
            auc_preds.append(scores_np[np.array([g2l[u] for u in vu]), ei1_np[in_chunk]])
            auc_labels.append(el_np[in_chunk])

        # Mask training edges and find top-K
        scores_t = torch.from_numpy(scores_np)
        mask_rows = torch.from_numpy(mask_sparse[chunk].toarray().astype(np.bool_))
        scores_t[mask_rows] = -float('inf')
        del scores_np, mask_rows
        _, topk = torch.topk(scores_t, k=k, dim=1)
        del scores_t

        # Remap positive val edges to chunk-local user indices
        in_chunk_pos = np.isin(pos_ei0, chunk)
        if not np.any(in_chunk_pos):
            continue
        b_ei = pos_ei[:, in_chunk_pos].clone()
        b_ei[0] = torch.tensor([g2l[int(u)] for u in b_ei[0]])

        g_recall.update(topk, b_ei)
        g_ndcg.update(topk, b_ei)
        g_mrr.update(topk, b_ei)

        chunk_deg = user_degrees[chunk]
        for bi, (low, high) in enumerate(buckets):
            in_bucket = set(np.where((chunk_deg > low) & (chunk_deg <= high))[0].tolist())
            bm = np.array([int(u) in in_bucket for u in b_ei[0].numpy()])
            if np.any(bm):
                bb_ei = b_ei[:, bm]
                b_counts[bi] += len(torch.unique(bb_ei[0]))
                b_recall[bi].update(topk, bb_ei)
                b_ndcg[bi].update(topk, bb_ei)
                b_mrr[bi].update(topk, bb_ei)

    auc = roc_auc_score(np.concatenate(auc_labels), np.concatenate(auc_preds))
    metrics = {
        'recall': g_recall.compute().item(),
        'ndcg':   g_ndcg.compute().item(),
        'mrr':    g_mrr.compute().item(),
        'auc':    auc,
    }
    bucketed = np.array([
        [b_recall[i].compute().item(), b_ndcg[i].compute().item(), b_mrr[i].compute().item()]
        for i in range(len(buckets))
    ])
    return metrics, bucketed, b_counts


def eval_gnn(model, name):
    """Helper: evaluate a GNN model and add results to the shared dicts."""
    global final_counts
    print(f"\nEvaluating {name}...")
    u_emb, i_emb = model.get_user_item_embeddings(train_data)
    i_emb_T = i_emb.T
    score_fn = lambda chunk: (u_emb[chunk] @ i_emb_T).astype(np.float32)
    m, bucketed, counts = chunked_evaluate(val_data, score_fn, gnn_train_sparse, train_user_degrees)
    results.append({'name': name, **m})
    all_plot_data[name] = bucketed
    final_counts = counts
    del u_emb, i_emb, i_emb_T
    gc.collect()
    print(f"  Recall={m['recall']:.4f}  NDCG={m['ndcg']:.4f}  MRR={m['mrr']:.4f}  AUC={m['auc']:.4f}")


# ── Shared setup ──────────────────────────────────────────────────────────────
mp_edge_index = train_data["user", "purchased", "item"].edge_index
gnn_train_sparse = sp.csr_matrix(
    (np.ones(mp_edge_index.shape[1]),
     (mp_edge_index[0].numpy(), mp_edge_index[1].numpy())),
    shape=(num_users, num_items)
)
train_user_degrees = np.array(gnn_train_sparse.sum(axis=1)).flatten()

# ── Run all methods ───────────────────────────────────────────────────────────
results = []
all_plot_data = {}
final_counts = None

for method in ['preferential_attachment', 'jaccard', 'resource_allocation', 'heat_conduction']:
    print(f"\nEvaluating {method}...")
    predictor = ClassicalGraphPredictor(num_users, num_items, method=method)
    predictor.fit(train_full_edges)
    m, bucketed, counts = chunked_evaluate(val_data, predictor.predict_chunk, predictor.R_train, train_user_degrees)
    results.append({'name': method, **m})
    all_plot_data[method] = bucketed
    final_counts = counts
    del predictor
    gc.collect()
    print(f"  Recall={m['recall']:.4f}  NDCG={m['ndcg']:.4f}  MRR={m['mrr']:.4f}  AUC={m['auc']:.4f}")

eval_gnn(gnn_model, 'gnn_bce')

try:
    eval_gnn(gnn_bpr, 'gnn_bpr')
except NameError:
    print("\ngnn_bpr not found — run the BPR cell first to include it.")

try:
    eval_gnn(gnn_playtime, 'gnn_playtime')
except NameError:
    print("\ngnn_playtime not found — run the playtime cell first to include it.")

# ── Results table ─────────────────────────────────────────────────────────────
print()
header = f"{'Model':<25} | {f'Recall@{K_VAL}':<10} | {f'NDCG@{K_VAL}':<10} | {f'MRR@{K_VAL}':<10} | {'AUC':<10}"
print(header)
print('=' * len(header))
for r in results:
    print(f"{r['name']:<25} | {r['recall']:<10.4f} | {r['ndcg']:<10.4f} | {r['mrr']:<10.4f} | {r['auc']:<10.4f}")
print('=' * len(header))


## 9. Extended Training (20 Epochs)

Continue BCE training for 15 more epochs then re-evaluate the GNN.

In [ ]:
EXTENDED_MODEL_PATH = '/content/gnn_model_20ep.pth'
EXTRA_EPOCHS = 15
LR = 0.001
BATCH_SIZE = 128

if os.path.exists(EXTENDED_MODEL_PATH):
    print(f"Found {EXTENDED_MODEL_PATH} — loading, skipping extended training.")
    gnn_model.load_state_dict(torch.load(EXTENDED_MODEL_PATH, map_location=device))
    gnn_model.eval()

else:
    print("Continuing BCE training (epochs 6–20)...")
    optimizer_ext = torch.optim.Adam(gnn_model.parameters(), lr=LR)
    train_loader_ext = LinkNeighborLoader(
        data=train_data,
        num_neighbors=[20, 10],
        neg_sampling_ratio=2.0,
        edge_label_index=(("user", "purchased", "item"), train_data["user", "purchased", "item"].edge_label_index),
        edge_label=train_data["user", "purchased", "item"].edge_label,
        batch_size=BATCH_SIZE,
        shuffle=True,
    )

    for epoch in range(6, 6 + EXTRA_EPOCHS):
        total_loss = total_examples = 0
        gnn_model.train()
        for sampled_data in tqdm.tqdm(train_loader_ext, desc=f"Epoch {epoch:03d}"):
            optimizer_ext.zero_grad()
            sampled_data.to(device)
            pred = gnn_model(sampled_data)
            ground_truth = sampled_data["user", "purchased", "item"].edge_label
            loss = F.binary_cross_entropy_with_logits(pred, ground_truth)
            loss.backward()
            optimizer_ext.step()
            total_loss += loss.item() * pred.numel()
            total_examples += pred.numel()
        print(f"Epoch: {epoch:03d}, Loss: {total_loss / total_examples:.4f}")

    torch.save(gnn_model.state_dict(), EXTENDED_MODEL_PATH)
    print(f"Saved to {EXTENDED_MODEL_PATH}")

# ── Re-evaluate GNN (20 epochs) — baselines unchanged, reuse results ──────────
print("\nEvaluating GNN (20 epochs)...")
user_emb_np, item_emb_np = gnn_model.get_user_item_embeddings(train_data)
item_emb_T = item_emb_np.T
gnn_score_fn_20 = lambda chunk: (user_emb_np[chunk] @ item_emb_T).astype(np.float32)

m_gnn20, bucketed_gnn20, _ = chunked_evaluate(
    val_data, gnn_score_fn_20, gnn_train_sparse, train_user_degrees
)
del user_emb_np, item_emb_np, item_emb_T
gc.collect()
print(f"  Recall={m_gnn20['recall']:.4f}  NDCG={m_gnn20['ndcg']:.4f}  MRR={m_gnn20['mrr']:.4f}  AUC={m_gnn20['auc']:.4f}")

all_plot_data['gnn_20ep'] = bucketed_gnn20

all_results_20ep = results + [{'name': 'gnn_20ep', **m_gnn20}]
print()
header = f"{'Model':<25} | {f'Recall@{K_VAL}':<10} | {f'NDCG@{K_VAL}':<10} | {f'MRR@{K_VAL}':<10} | {'AUC':<10}"
print(header)
print('=' * len(header))
for r in all_results_20ep:
    print(f"{r['name']:<25} | {r['recall']:<10.4f} | {r['ndcg']:<10.4f} | {r['mrr']:<10.4f} | {r['auc']:<10.4f}")
print('=' * len(header))


## 10. Results Visualisation

Per-user-degree bucket plots for all methods.

In [ ]:
import matplotlib.pyplot as plt

metrics_info = [('Recall', 0), ('NDCG', 1), ('MRR', 2)]
base_labels = ['0', '1', '2-5', '6-10', '11-50', '51+']
bucket_labels_with_counts = [f"{lbl}\n(n={cnt})" for lbl, cnt in zip(base_labels, final_counts)]
methods = list(all_plot_data.keys())

n_buckets = len(bucket_labels_with_counts)
n_methods = len(methods)
width = 0.12
gap = 0.5
group_width = n_methods * width + gap
x = np.arange(n_buckets) * group_width

for metric_name, m_idx in metrics_info:
    fig, ax = plt.subplots(figsize=(14, 6))
    for i, method in enumerate(methods):
        offset = (i - n_methods / 2 + 0.5) * width
        ax.bar(x + offset, all_plot_data[method][:, m_idx], width, label=method)

    for xi in x[1:]:
        ax.axvline(xi - group_width / 2, color='gray', linestyle=':', linewidth=1.0, alpha=0.6)

    ax.set_title(f'{metric_name}@{K_VAL} by User Training Degree')
    ax.set_xticks(x)
    ax.set_xticklabels(bucket_labels_with_counts)
    ax.set_xlabel('User Degree Bucket (n = users evaluated)')
    ax.set_ylabel('Score')
    ax.legend(loc='upper left', ncol=2)
    ax.grid(axis='y', linestyle='--', alpha=0.7)

    plt.tight_layout()
    fname = f'bucketed_{metric_name.lower()}.png'
    plt.savefig(fname, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Saved {fname}")
